In [1]:
import os
import pandas as pd
import pydata_google_auth
from google.cloud import bigquery
from dotenv import load_dotenv

In [2]:
# Carregar variáveis de ambiente (ajuste o caminho relativo conforme a estrutura)
load_dotenv("../.env")
PROJECT_ID = os.getenv("GCP_PROJECT_ID", "seu-project-id")

print(f"Projeto configurado: {PROJECT_ID}")

Projeto configurado: conductive-coil-502322-f0


In [3]:
# Reutilizando a lógica de autenticação do executar_sql.py
credentials = pydata_google_auth.get_user_credentials(
    scopes=["https://www.googleapis.com/auth/cloud-platform"],
)

# Iniciar client do BigQuery
client = bigquery.Client(
    project=PROJECT_ID, 
    credentials=credentials.with_quota_project(PROJECT_ID)
)
print("Conectado ao BigQuery com sucesso!")

Conectado ao BigQuery com sucesso!


In [4]:
sql_pipeline_fase3 = f"""
-- ============================================================================
-- 1) BASE ANALÍTICA EM NÍVEL DE ALUNO
-- ============================================================================
CREATE OR REPLACE TABLE `{PROJECT_ID}.gold.alunos_features`
PARTITION BY RANGE_BUCKET(ano, GENERATE_ARRAY(2020, 2035, 1))
CLUSTER BY sigla_uf, rede AS

WITH alunos_validos AS (
  SELECT *
  FROM `{PROJECT_ID}.silver.alunos`
  WHERE proficiencia IS NOT NULL
    AND LOWER(COALESCE(presenca, '')) IN ('presente', '1')
),

meta_ano AS (
  SELECT
    id_municipio,
    ano_meta,
    LOWER(rede)          AS rede_norm,
    ANY_VALUE(meta_taxa) AS meta_taxa
  FROM `{PROJECT_ID}.silver.metas_municipio`
  WHERE meta_taxa IS NOT NULL
  GROUP BY id_municipio, ano_meta, rede_norm
)

SELECT
  a.id_aluno,
  a.id_escola,
  a.id_municipio,
  a.alfabetizado AS y_alfabetizado_label,
  CASE
    WHEN LOWER(a.alfabetizado) IN ('sim', '1') THEN 1
    WHEN LOWER(a.alfabetizado) IN ('não', 'nao', '0') THEN 0
  END AS y_alfabetizado,
  a.ano,
  a.serie,
  a.rede,
  m.sigla_uf,
  m.nome_regiao,
  m.nome_municipio,
  COUNT(*)           OVER (PARTITION BY a.ano, a.id_escola)    AS qtd_alunos_escola,
  COUNT(*)           OVER (PARTITION BY a.ano, a.id_municipio) AS qtd_alunos_municipio,
  COUNT(DISTINCT a.id_escola)
                     OVER (PARTITION BY a.ano, a.id_municipio) AS qtd_escolas_municipio,
  a.peso_aluno AS w_peso_aluno,
  mt.meta_taxa AS ref_meta_taxa_ano,
  a.proficiencia AS leak_proficiencia
FROM alunos_validos a
LEFT JOIN `{PROJECT_ID}.silver.dim_municipio` m
  ON a.id_municipio = m.id_municipio
LEFT JOIN meta_ano mt
  ON a.id_municipio = mt.id_municipio
 AND a.ano          = mt.ano_meta
 AND LOWER(a.rede)  = mt.rede_norm;


-- ============================================================================
-- 2) DICIONÁRIO DE COLUNAS (Governança)
-- ============================================================================
CREATE OR REPLACE TABLE `{PROJECT_ID}.governanca.dicionario_features_fase3` AS
SELECT * FROM UNNEST([
  STRUCT('id_aluno' AS coluna, 'identificador' AS papel, 'Chave do aluno; rastreabilidade' AS observacao),
  ('id_escola',             'identificador', 'Chave INEP da escola; usar como grupo em GroupKFold'),
  ('id_municipio',          'identificador', 'Código IBGE 7 dígitos; alta cardinalidade (~5.570)'),
  ('y_alfabetizado',        'target',        'Alvo binário 1=Sim / 0=Não'),
  ('y_alfabetizado_label',  'target',        'Alvo original decodificado'),
  ('ano',                   'split',         'Define treino (2023) e teste (2024); não é feature'),
  ('serie',                 'feature_cat',   'Baixa cardinalidade — one-hot'),
  ('rede',                  'feature_cat',   'Baixa cardinalidade — one-hot'),
  ('sigla_uf',              'feature_cat',   '27 níveis — one-hot ou target encoding no pipeline'),
  ('nome_regiao',           'feature_cat',   '5 níveis — one-hot'),
  ('nome_municipio',        'rotulo',        'Apenas relatório; na modelagem usar id_municipio'),
  ('qtd_alunos_escola',     'feature_num',   'Porte da escola no ano'),
  ('qtd_alunos_municipio',  'feature_num',   'Porte do município no ano'),
  ('qtd_escolas_municipio', 'feature_num',   'Rede escolar do município no ano'),
  ('w_peso_aluno',          'peso',          'sample_weight; nunca feature'),
  ('ref_meta_taxa_ano',     'bloqueada',     'Meta derivada da taxa-base do município'),
  ('leak_proficiencia',     'bloqueada',     'Define o target: alfabetizado := proficiencia>=743')
]);


-- ============================================================================
-- 3) CRIAÇÃO DA AMOSTRA ESTRATIFICADA (~5%)
-- ============================================================================
CREATE OR REPLACE TABLE `{PROJECT_ID}.gold.alunos_features_amostra`
CLUSTER BY sigla_uf, rede AS
SELECT *
FROM `{PROJECT_ID}.gold.alunos_features`
WHERE MOD(ABS(FARM_FINGERPRINT(id_aluno)), 20) = 0;
"""

print("Executando o script SQL no BigQuery (isso pode levar alguns segundos)...")
job = client.query(sql_pipeline_fase3)
job.result() # Aguarda a finalização de todos os comandos DDL
print("Tabelas Gold e Amostra criadas com sucesso no BigQuery!")

Executando o script SQL no BigQuery (isso pode levar alguns segundos)...
Tabelas Gold e Amostra criadas com sucesso no BigQuery!


In [5]:
# Baixando o dicionário
df_dicionario = client.query(f"SELECT * FROM `{PROJECT_ID}.governanca.dicionario_features_fase3`").to_dataframe()
print("--- Dicionário de Features Carregado ---")
display(df_dicionario.head())

# Baixando a ABT amostral
sql_abt = f"SELECT * FROM `{PROJECT_ID}.gold.alunos_features_amostra`"
print("\nBaixando a ABT amostral para o Pandas...")
df_abt = client.query(sql_abt).to_dataframe()

print(f"\nBase carregada com sucesso! Linhas: {df_abt.shape[0]} | Colunas: {df_abt.shape[1]}")

print("\nDistribuição do Target (y_alfabetizado):")
print(df_abt['y_alfabetizado'].value_counts(normalize=True).round(4) * 100)

display(df_abt.head())

c:\Users\felyp\OneDrive\Área de Trabalho\faculdade\tech-challenge-machine-learning\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


--- Dicionário de Features Carregado ---


,coluna,papel,observacao
0,y_alfabetizado_label,target,Alvo original decodificado
1,w_peso_aluno,peso,sample_weight; nunca feature
2,ref_meta_taxa_ano,bloqueada,Meta derivada da taxa-base do município
3,nome_regiao,feature_cat,5 níveis — one-hot
4,sigla_uf,feature_cat,27 níveis — one-hot ou target encoding no pipe...



Baixando a ABT amostral para o Pandas...


c:\Users\felyp\OneDrive\Área de Trabalho\faculdade\tech-challenge-machine-learning\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(



Base carregada com sucesso! Linhas: 167791 | Colunas: 17

Distribuição do Target (y_alfabetizado):
y_alfabetizado
1    59.14
0    40.86
Name: proportion, dtype: Float64


,id_aluno,id_escola,id_municipio,y_alfabetizado_label,y_alfabetizado,ano,serie,rede,sigla_uf,nome_regiao,nome_municipio,qtd_alunos_escola,qtd_alunos_municipio,qtd_escolas_municipio,w_peso_aluno,ref_meta_taxa_ano,leak_proficiencia
0,11019605,60000159,1100015,Sim,1,2023,2° ano do Ensino Fundamental,Municipal,RO,Norte,Alta Floresta D'Oeste,24,227,5,1.083333,NaN,802.969529
1,11019624,60000161,1100015,Sim,1,2023,2° ano do Ensino Fundamental,Municipal,RO,Norte,Alta Floresta D'Oeste,45,227,5,1.127451,NaN,799.911558
2,11019780,60000382,1100015,Sim,1,2023,2° ano do Ensino Fundamental,Municipal,RO,Norte,Alta Floresta D'Oeste,70,227,5,0.971731,NaN,841.216969
3,11019589,60000159,1100015,Sim,1,2023,2° ano do Ensino Fundamental,Municipal,RO,Norte,Alta Floresta D'Oeste,24,227,5,1.083333,NaN,802.969529
4,11019737,60000337,1100015,Não,0,2023,2° ano do Ensino Fundamental,Municipal,RO,Norte,Alta Floresta D'Oeste,84,227,5,1.163368,NaN,704.503154


In [6]:
# Criar diretório se não existir
os.makedirs("../data/raw", exist_ok=True)

# Salvar o DataFrame
caminho_arquivo = "../data/raw/abt_alunos_alfabetizacao.parquet"
df_abt.to_parquet(caminho_arquivo, index=False)

print(f"Arquivo salvo com sucesso em: {caminho_arquivo}")

Arquivo salvo com sucesso em: ../data/raw/abt_alunos_alfabetizacao.parquet
